# GraphRAG V2 — topic-agnostic knowledge-graph RAG

Improved version of [graphrag_ai_copyright.ipynb](./graphrag_ai_copyright.ipynb). The pipeline logic lives in
[graphrag_pipeline.py](./graphrag_pipeline.py); **all topic-specific inputs live in one JSON file** under
[topics/](./topics) (explained in [PIPELINE_INPUTS.md](./PIPELINE_INPUTS.md)). To research another topic you only
change `CONFIG_FILE` below.

### Before you run
1. **Kernel:** select **`GraphRAG (Python 3.12)`** (top-right kernel picker). If it is missing run `bash setup_env.sh`
   in a terminal. The default Codespaces Python (3.14) cannot install `graspologic` — that was the package conflict in V1.
2. **Credentials:** add `OPENAI_API_KEY` (and `SERPAPI` / `SERPAPI_API_KEY` for new topics) as
   [GitHub Codespaces secrets](https://github.com/settings/codespaces) with access to this repository, then restart the
   codespace. A local `.env` file also works. Keys are never printed.
3. **Cost:** only steps 4, 6 and 8 call OpenAI. Results are cached in `outputs/<output_name>/`, so re-running a cell
   does **not** pay twice.

| Step | What happens | Calls OpenAI? |
|---|---|---|
| 1 | Load topic config + credentials | – |
| 2 | Load CSV *or* collect articles with SerpAPI + Trafilatura | – (SerpAPI in search mode) |
| 3 | Split articles into sentence-aware chunks, estimate cost | – |
| 4 | Ontology-constrained entity/relationship extraction (cached per chunk) | ✅ `extraction_model` |
| 5 | Merge entities across chunks into one directed graph | – |
| 6 | Hierarchical Leiden communities + LLM community reports (cached) | ✅ `extraction_model` |
| 7 | Interactive D3 visualization | – |
| 8 | Map-reduce querying with `[S#]` source citations | ✅ both models |

In [1]:
import sys
import warnings

warnings.filterwarnings("ignore", category=SyntaxWarning)
if sys.version_info >= (3, 13):
    raise RuntimeError(
        f"This kernel runs Python {sys.version.split()[0]}, but graspologic needs Python <= 3.12.\n"
        "Run `bash setup_env.sh` in a terminal, then select the 'GraphRAG (Python 3.12)' kernel."
    )

import pandas as pd
from IPython.display import HTML, Markdown, display

import graphrag_pipeline as gp

pd.set_option("display.max_colwidth", 120)
print(f"✅ Python {sys.version.split()[0]} — imports ready")

✅ Python 3.12.3 — imports ready


---
## 1. Choose the topic config

`topics/ai_copyright.json` reproduces V1 (same dataset and ontology, plus a `DEVELOPS` relation linking companies to
their AI systems). For a new topic copy [topics/TEMPLATE.json](./topics/TEMPLATE.json) — see
[topics/ev_battery_recycling.json](./topics/ev_battery_recycling.json) for a filled-in search-mode example.

In [2]:
CONFIG_FILE = "topics/ev_battery_recycling.json"   # ← the only line to change for another topic

config = gp.load_config(CONFIG_FILE)
print(gp.config_summary(config))

creds = gp.load_credentials()  # Codespaces secrets → env vars; .env as fallback
print(f"\nCredentials: OpenAI={'✅ found' if creds['openai'] else '❌ missing'}, "
      f"SerpAPI={'✅ found' if creds['serpapi'] else '— not found (only needed for input_mode=search)'}")

pipeline = gp.GraphRAGPipeline(config)

Topic:       Electric vehicle battery recycling and second-life batteries
Output dir:  outputs/ev_battery_recycling
Input mode:  search → outputs/ev_battery_recycling/dataset.csv
Models:      extraction/summaries/map=gpt-4o-mini, final answers=gpt-4o
Ontology:    8 entity types, 9 relationship types
Questions:   3
Queries:     ['EV battery recycling companies', 'lithium-ion battery recycling regulation'] (6 results each)

Credentials: OpenAI=✅ found, SerpAPI=✅ found


### The ontology

The ontology is the **schema of the knowledge graph**: the LLM may only use these entity and relationship types
(enforced by Pydantic `Literal` types built from the config). The descriptions are sent to the LLM, so precise
descriptions with examples give a cleaner graph.

In [3]:
display(pd.DataFrame(config.entity_types.items(), columns=["entity type", "meaning"]))
display(pd.DataFrame(config.relation_types.items(), columns=["relationship type (source → target)", "meaning"]))

,entity type,meaning
0,ORGANIZATION,"Companies, start-ups, OEMs, research institutes (e.g. Redwood Materials, Tesla)"
1,PERSON,"Executives, researchers, policymakers"
2,GOVERNMENT,"Countries, regulators and agencies (e.g. European Commission, US DOE)"
3,REGULATION,"Laws, directives, standards, subsidies (e.g. EU Battery Regulation, Inflation Reduction Act)"
4,TECHNOLOGY,"Recycling or battery technologies (e.g. hydrometallurgy, direct recycling, LFP)"
5,MATERIAL,"Raw or recovered materials (e.g. lithium, cobalt, nickel, black mass)"
6,FACILITY,"Plants, gigafactories, recycling sites"
7,CONCEPT,"Ideas, markets and metrics (e.g. circular economy, recovery rate, second-life use)"


,relationship type (source → target),meaning
0,OPERATES,ORGANIZATION runs a FACILITY
1,DEVELOPS,ORGANIZATION/PERSON develops or uses a TECHNOLOGY
2,RECOVERS,TECHNOLOGY/FACILITY/ORGANIZATION recovers a MATERIAL
3,SUPPLIES,ORGANIZATION supplies MATERIAL or batteries to another ORGANIZATION
4,PARTNERS_WITH,ORGANIZATION collaborates or has a deal with another ORGANIZATION
5,REGULATES,"GOVERNMENT/REGULATION sets rules or targets for an ORGANIZATION, MATERIAL or CONCEPT"
6,FUNDS,GOVERNMENT/ORGANIZATION finances an ORGANIZATION or FACILITY
7,PART_OF,PERSON/ORGANIZATION belongs to an ORGANIZATION/GOVERNMENT
8,LOCATED_IN,FACILITY/ORGANIZATION is located in a GOVERNMENT (country/region)


---
## 2. Load the documents

* `input_mode = "csv"` reads `dataset_file` (only a `full_text` column is required).
* `input_mode = "search"` runs the SerpAPI search once, scrapes the pages with Trafilatura, drops short/off-topic pages
  (`topic_terms`) and saves `outputs/<output_name>/dataset.csv`. Later runs reuse that file; pass `refresh=True`
  to scrape again.

Every document gets a source id (`S1`, `S2`, …) that is used for citations in summaries and answers.

In [4]:
df = pipeline.load_documents(refresh=False)
df[["source_id", "title", "source", "url"]]

📂 Using existing dataset outputs/ev_battery_recycling/dataset.csv
✅ 5 documents ready (of 5 rows; max_articles=8)


,source_id,title,source,url
0,S1,Used Lithium-Ion Batteries,U.S. Environmental Protection Agency (.gov),https://www.epa.gov/recycle/used-lithium-ion-batteries
1,S2,Complete Guide to Battery Recycling Regulations & ...,Green Li-ion,https://www.greenli-ion.com/post/battery-recycling-regulations-compliance-guide-2caa3
2,S3,Battery Stewardship - CalRecycle Home Page - CA.gov,CalRecycle Home Page (.gov),https://calrecycle.ca.gov/epr/batteries/
3,S4,State Recycling Laws,Battery Council International,https://batterycouncil.org/policy-initiatives/state-recycling-laws/
4,S5,Responsible Battery Recycling Act of 2022,Cirba Solutions,https://www.cirbasolutions.com/responsible-battery-recycling-act-of-2022/?srsltid=AYCX9vIupWP-xWJNr8Qv4XtrFac5e8rTqj...


---
## 3. Chunk the documents

V1 sent whole articles (up to ~45k characters) in one call, so the LLM skimmed long articles and missed entities.
V2 splits them into overlapping, sentence-aware chunks of `chunk_size` tokens.

In [5]:
chunks = pipeline.chunk()
pipeline.estimate_calls()

✅ 14 chunks from 5 documents (0 already extracted in cache, 14 need LLM calls)
⚠️  1 documents truncated by max_chunks_per_article=6


{'extraction_calls': 14,
 'approx_input_tokens_for_extraction': 15612,
 'questions': 3}

---
## 4. Extract entities and relationships  💲

Each chunk is sent to `extraction_model` with the ontology-constrained prompt. Results are appended to
`outputs/<output_name>/extraction_cache.jsonl` immediately, so an interrupted run resumes where it stopped and
failed chunks are retried on the next run.

In [6]:
gp.load_credentials(require_openai=True)
extractions = await pipeline.aextract()

🧠 Extracting 14 chunks with gpt-4o-mini (0 cached)...


Extracting: 100%|██████████| 14/14 [00:51<00:00,  3.66s/it]

✅ Extractions available for 14/14 chunks


In [7]:
# Inspect the chunk with the most extracted entities
sample = max(extractions.values(), key=lambda e: len(e["entities"]))
print("Source:", sample["source_id"], "—", pipeline.sources[sample["source_id"]]["title"])
display(pd.DataFrame(sample["entities"]))
display(pd.DataFrame(sample["relationships"]))

Source: S2 — Complete Guide to Battery Recycling Regulations & ...


,name,type,description
0,Environmental Protection Agency,GOVERNMENT,The Environmental Protection Agency (EPA) is a U.S. government agency responsible for regulating and overseeing envi...
1,Bipartisan Infrastructure Law,REGULATION,The Bipartisan Infrastructure Law allocates funding for developing battery collection best practices and guidelines ...
2,New Jersey,GOVERNMENT,New Jersey is a U.S. state that has implemented comprehensive electric vehicle battery management requirements throu...
3,EU Battery Regulation 2023/1542,REGULATION,The EU Battery Regulation 2023/1542 establishes a comprehensive framework for battery sustainability and recycling c...
4,California,GOVERNMENT,"California is a U.S. state known for its stringent battery recycling regulations, including stewardship programs for..."
5,Washington State,GOVERNMENT,"Washington State has enacted SB 5144, requiring battery producers to join Battery Stewardship Organizations and impl..."
6,Connecticut,GOVERNMENT,Connecticut is a U.S. state that has implemented producer responsibility frameworks for battery recycling.
7,Colorado,GOVERNMENT,Colorado is a U.S. state that has adopted similar producer responsibility laws for battery recycling.
8,Nebraska,GOVERNMENT,Nebraska is a U.S. state that has implemented battery recycling regulations aligned with producer responsibility mod...
9,Illinois,GOVERNMENT,Illinois is a U.S. state that has enacted battery recycling regulations similar to those in other states with produc...


,source,target,relation,description
0,Environmental Protection Agency,Bipartisan Infrastructure Law,FUNDS,The Environmental Protection Agency is funded by the Bipartisan Infrastructure Law to develop battery collection bes...
1,New Jersey,Electric and Hybrid Vehicle Battery Management Act,REGULATES,New Jersey regulates electric vehicle battery management through its Electric and Hybrid Vehicle Battery Management ...
2,EU Battery Regulation 2023/1542,lithium,REGULATES,The EU Battery Regulation 2023/1542 sets recycling targets for lithium as part of its comprehensive battery manageme...
3,California,battery stewardship,REGULATES,California implements battery stewardship regulations to ensure responsible recycling and disposal of batteries.
4,Washington State,battery stewardship,REGULATES,Washington State's SB 5144 mandates battery producers to participate in battery stewardship organizations.
5,Connecticut,producer responsibility,REGULATES,Connecticut has established producer responsibility laws for battery recycling.
6,Colorado,producer responsibility,REGULATES,Colorado has implemented producer responsibility frameworks for battery recycling.
7,Nebraska,producer responsibility,REGULATES,Nebraska has adopted producer responsibility regulations for battery recycling.
8,Illinois,producer responsibility,REGULATES,Illinois has enacted producer responsibility laws similar to those in other states.
9,Texas,battery recycling regulations,REGULATES,Texas maintains less stringent battery recycling regulations compared to other states.


---
## 5. Build the knowledge graph

Entities from all chunks are merged by normalised name (`"OpenAI"`, `"OPENAI"`, `"The OpenAI"` → one node). Unlike
V1, all descriptions and source ids are kept instead of being overwritten by the last chunk, relationship direction
is preserved, and relationships pointing at entities that were never extracted are dropped.

In [8]:
graph = pipeline.build_graph()

entities = pd.DataFrame(
    [{"entity": d["name"], "type": d["type"], "connections": graph.degree(n), "sources": ", ".join(d["sources"])}
     for n, d in graph.nodes(data=True)]
).sort_values("connections", ascending=False)
display(entities["type"].value_counts().to_frame("entities"))
entities.head(15)

✅ Graph: 91 entities, 123 relationships (6 relationships with unknown endpoints dropped)


,entities
type,
GOVERNMENT,30
CONCEPT,21
MATERIAL,13
REGULATION,13
ORGANIZATION,7
TECHNOLOGY,5
FACILITY,2


,entity,type,connections,sources
0,lithium-ion batteries,MATERIAL,25,"S1, S2, S5"
72,BCI,ORGANIZATION,12,S4
2,EPA,GOVERNMENT,11,"S1, S2"
25,California,GOVERNMENT,8,"S2, S3, S5"
39,recycling technologies,TECHNOLOGY,8,S2
69,CalRecycle,ORGANIZATION,7,S3
70,Responsible Battery Recycling Act,REGULATION,7,S3
73,lead-acid battery,MATERIAL,7,S4
3,Call2Recycle,ORGANIZATION,6,"S1, S2"
33,lithium,MATERIAL,5,"S2, S3"


In [9]:
# Look up one entity: its descriptions and relationships
inspect_name = entities.iloc[0]["entity"]   # e.g. "OpenAI"

key = gp.normalize_name(inspect_name)
if key not in graph:
    print(f"'{inspect_name}' is not in the graph")
else:
    node = graph.nodes[key]
    print(f"{node['name']} [{node['type']}] — sources {node['sources']}")
    for description in node["descriptions"]:
        print("  •", description)
    for u, v, data in list(graph.out_edges(key, data=True)) + list(graph.in_edges(key, data=True)):
        print(f"  {graph.nodes[u]['name']} --[{data['relation']}]--> {graph.nodes[v]['name']}")

lithium-ion batteries [MATERIAL] — sources ['S1', 'S2', 'S5']
  • Lithium-ion batteries are rechargeable batteries used in various applications, including electric vehicles, and require proper recycling to recover critical materials.
  • Lithium-ion batteries are rechargeable batteries commonly used in consumer electronics and electric vehicles, which require proper recycling at end-of-life.
  • Lithium-ion batteries are rechargeable batteries commonly used in consumer electronics and electric vehicles.
  • Lithium-ion batteries are a type of rechargeable battery that requires specialized handling due to their size and material complexity.
  • A type of battery that is being targeted for recycling under the Responsible Battery Recycling Act.
  lithium-ion batteries --[SUPPLIES]--> U.S. Geological Survey
  lithium-ion batteries --[REGULATES]--> EPA
  lithium-ion batteries --[PARTNERS_WITH]--> Call2Recycle
  lithium-ion batteries --[PART_OF]--> Avoid the Spark. Be Battery Safety Smart. c

---
## 6. Communities and community reports  💲

Hierarchical Leiden (seeded, so reproducible) groups closely connected entities into communities of at most
`max_cluster_size` entities. For each community the LLM writes a titled briefing with `[S#]` citations. These
reports are what the query engine reads, which enables big-picture questions that no single chunk can answer.
Reports are cached by their evidence, so unchanged communities are not re-summarised.

In [10]:
pipeline.detect_communities()
reports = await pipeline.asummarize()

for report in list(reports.values())[:5]:
    display(Markdown(f"**Community {report['community']} — {report['title']}** ({report['size']} entities)\n\n"
                     f"{report['summary']}"))

✅ 18 communities (+2 isolated entities)
✅ 18 community reports (18 new LLM calls, 0 from cache)


**Community 0 — Battery Stewardship and Regulatory Frameworks in EV Battery Recycling** (4 entities)

Battery stewardship programs are essential frameworks that ensure manufacturers finance and operate battery recycling systems, regulated by entities like the European Union and New York [S2]. The European Union's regulations serve as a global model, while New York mandates retailers to collect used batteries, reflecting a growing trend in the consumer goods sector to comply with these programs [S2]. However, there are ongoing discussions about the effectiveness and economic feasibility of these stewardship programs, as regulated by the EPA [S2]. Call2Recycle exemplifies a specific initiative within these programs, aiming to enhance recycling efforts [S2].

**Community 2 — Electric Vehicle Battery Recycling Initiatives** (3 entities)

The Environmental Protection Agency (EPA) plays a crucial role in regulating battery recycling initiatives in the U.S., supported by funding from the Bipartisan Infrastructure Law [S2]. This law allocates resources specifically for developing battery collection best practices, which aim to improve the efficiency and safety of recycling processes [S2]. While the EPA is tasked with creating these guidelines, there may be disputes regarding the effectiveness of current practices and the adequacy of funding, highlighting ongoing debates in the field of battery recycling [S2].

**Community 3 — California's Leadership in Battery Recycling and Stewardship** (9 entities)

California is at the forefront of battery recycling initiatives, implementing the Responsible Battery Recycling Act of 2022, which mandates manufacturers to establish recycling programs and stewardship plans for lithium-ion batteries [S5]. This act is regulated by the Department of Toxic Substances Control, ensuring compliance and oversight [S5]. In contrast, Washington State has enacted similar legislation requiring battery producers to join stewardship organizations, highlighting a regional commitment to battery lifecycle management [S2]. However, the effectiveness of these regulations is debated, particularly in comparison to the EU Battery Regulation 2023/1542, which sets stringent recycling targets for lithium [S2]. Organizations like Cirba Solutions play a crucial role in supporting these initiatives by enhancing recycling operations and collaborating with government entities [S5].

**Community 5 — Regulatory Framework for Lead-Acid Battery Recycling in the U.S.** (7 entities)

The community focuses on the regulation of lead-acid batteries and lead batteries, which are rechargeable batteries containing lead and sulfuric acid, widely used in vehicles [S4]. The BCI Model provides guidelines that influence both federal and state regulations, including State Recycling Laws that govern the recycling processes for these batteries [S4]. Kansas City, MO, and the New York Department of Environmental Conservation are key regulatory bodies enforcing these laws, although there may be disputes regarding the effectiveness and enforcement of these regulations [S4]. Overall, the interconnectedness of these entities highlights the complexity of battery recycling governance in the United States.

**Community 7 — Battery Passport and Recycling Compliance in EV Battery Management** (2 entities)

The Battery Passport concept [S2] serves as a framework for ensuring recycling compliance by tracking battery information throughout supply chains. This international coordination aims to enhance traceability and accountability in battery recycling processes. However, there are ongoing discussions about the effectiveness of the Battery Passport in achieving comprehensive compliance, with some stakeholders questioning its implementation and impact on actual recycling practices [S2].

---
## 7. Visualize the graph

Writes `outputs/<output_name>/<output_name>_graph.html` (and `graph_data.json`). Open the HTML file in a browser —
in Codespaces right-click it → *Download*, or open it with the *Live Preview* extension. Nodes are coloured by
entity type, the tooltip shows community and sources, and the sidebar filters types.

In [11]:
html_path = pipeline.visualize()

SHOW_INLINE = True  # set False if the inline preview does not render in your notebook UI
if SHOW_INLINE:
    import html as _html
    warnings.filterwarnings("ignore", message="Consider using IPython.display.IFrame")
    display(HTML(f'<iframe srcdoc="{_html.escape(html_path.read_text())}" '
                 'style="width:100%;height:650px;border:0"></iframe>'))

✅ Visualization: outputs/ev_battery_recycling/ev_battery_recycling_graph.html (open in a browser)


---
## 8. Query the knowledge graph  💲

**Map:** every community report is scored for relevance (0–10) and answered by `extraction_model`, in parallel.
**Reduce:** the `top_communities` most relevant partial answers are merged by `query_model` into one answer that keeps
the `[S#]` citations; the cited sources are listed underneath.

In [ ]:
for question in config.questions:
    result = await pipeline.aquery(question, verbose=True)
    display(Markdown(pipeline.format_answer(result)))

In [ ]:
# Try your own question
your_question = "What is the role of fair use in AI copyright disputes?"

display(Markdown(pipeline.format_answer(await pipeline.aquery(your_question, verbose=True))))

In [ ]:
pipeline.save_answers()  # writes outputs/<output_name>/answers.md

---
## 9. Come back later without rebuilding

Everything needed for querying is saved in `outputs/<output_name>/`. In a fresh kernel run the import cell and:

```python
pipeline = gp.GraphRAGPipeline(gp.load_config("topics/ai_copyright.json")).load_saved()
await pipeline.aquery("...")
```

or from a terminal: `.venv/bin/python run_pipeline.py --config topics/ai_copyright.json --ask "..."`.

### What changed from V1

| Area | V1 | V2 |
|---|---|---|
| Environment | Installed into Python 3.14 → `graspologic` conflict | Isolated Python 3.12 `.venv` + kernel (`setup_env.sh`), pinned lock file |
| Credentials | `.env` only | Codespaces secrets (`OPENAI_API_KEY`, `SERPAPI`…) with `.env` fallback; never printed |
| New topic | Edit many cells + duplicate ontology lists | One JSON config; ontology types generated dynamically |
| Long articles | Whole article per call | Sentence-aware chunks |
| Cost / re-runs | Every re-run pays again | Chunk- and community-level caches, resume after failure |
| Entity merging | Case-sensitive, last description wins | Normalised names, all descriptions + sources kept |
| Relationships | Undirected, label overwritten | Directed multigraph; dangling endpoints dropped |
| Communities | Mixed hierarchy levels, unseeded | Final Leiden level, seeded, isolates handled |
| Summaries / queries | Sequential, no citations | Parallel, relevance-ranked, `[S#]` citations + source list |
| Visualization | Hard-coded title/colours, unescaped HTML | Topic title/colours from config, XSS-safe |
| Bugs fixed | shared mutable `community_summaries`, crash in entity lookup cell when name missing | — |